# geo0221 – Coordinate Reference Systems in Code

**Unit:** geo0220 CRS & Projections · Slides: GeoInfo_CRS_Proj (Moodle)

A coordinate pair without its CRS is just two numbers. In this notebook you will
1. identify the CRS of four "mystery" files that all contain the **same DWD stations**,
2. transform between CRSs with `geopandas` / `pyproj` and verify the result numerically,
3. measure how wrong distances become in the wrong CRS,
4. handle the classic pitfalls: axis order, false easting with zone prefix, Web Mercator scale,
5. check everything visually in QGIS.

## 0. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
from pyproj import CRS, Transformer, Geod

DATA = Path("data")
sorted(p.name for p in DATA.glob("DWD_Stations_NRW_CRS_*.csv"))

## 1. Four files, one set of stations
### 1.1 Inspect before you read
**Predict:** which separators, date formats and coordinate columns do you expect?

In [ ]:
for p in sorted(DATA.glob("DWD_Stations_NRW_CRS_*.csv")):
    print(p.name)
    print("   ", p.read_text(encoding="utf-8").splitlines()[0])
    print("   ", p.read_text(encoding="utf-8").splitlines()[1])

### 1.2 Exercise: read all four files into DataFrames
File A differs from B–D (separator, column names, quoting). Write a small function `read_station_csv(path)` that returns a DataFrame with the columns `station_id, name, x, y` regardless of the file variant.

In [ ]:
# YOUR CODE HERE

dfs = {p.stem[-1]: read_station_csv(p) for p in sorted(DATA.glob("DWD_Stations_NRW_CRS_*.csv"))}
pd.concat({k: v.head(2) for k, v in dfs.items()})

In [ ]:
# Value ranges are the strongest hint for the CRS
pd.DataFrame({k: v[["x", "y"]].agg(["min", "max"]).unstack() for k, v in dfs.items()}).T

### 1.3 The mystery
For each file, decide on an EPSG code and justify it in one sentence. Hints: degrees? metres? how big are the numbers? what is special about C compared with D?
Use [epsg.io](https://epsg.io) to compare candidates.

| File | EPSG | Reason |
|---|---|---|
| A | | |
| B | | |
| C | | |
| D | | |

In [ ]:
# YOUR CODE HERE
for k, epsg in CANDIDATES.items():
    print(k, epsg, CRS(epsg).name)

## 2. Verify, don't guess
### 2.1 Build GeoDataFrames and transform everything to EPSG:25832
**Watch out:** `points_from_xy(x, y)` expects *x = easting/longitude*, *y = northing/latitude*.

In [ ]:
# YOUR CODE HERE
{k: g.crs.to_epsg() for k, g in gdfs.items()}

### 2.2 How close are the transformed coordinates to file D?
If the guesses are right, the deviations should be at the millimetre level.

In [ ]:
ref = gdfs["D"]
for k, g in gdfs.items():
    g = g.loc[ref.index]
    dev = g.geometry.distance(ref.geometry)
    print(f"{k}: max deviation {dev.max():.4f} m")

**Discuss:** what would the deviation look like if you had chosen EPSG:32632 (WGS 84 / UTM 32N) instead of EPSG:25832 for D? Try it. Why is the difference so small in practice, and when does it matter?

## 3. Distances in the wrong CRS
### 3.1 Reference: geodesic distance on the ellipsoid

In [ ]:
geod = Geod(ellps="GRS80")
a, b = dfs["A"].loc[216], dfs["A"].loc[389]
_, _, d_geod = geod.inv(a.x, a.y, b.x, b.y)
print(f"{a['name']} → {b['name']}: {d_geod:,.1f} m (geodesic)")

### 3.2 Exercise: compute the same distance naively in each original CRS
Use the plain Euclidean distance between the two points **in the original coordinates** of A, B, C and D and compare with the geodesic reference.
**Predict first:** which one is wrong, and by roughly which factor?

In [ ]:
# YOUR CODE HERE
pd.DataFrame(rows)

**Interpretation:**
* A: the result is in **degrees**, which is meaningless as a distance.
* B: Web Mercator stretches distances by about 1/cos(latitude) ≈ 1.6 at 51.5° N. Fine for web tiles, wrong for measurements.
* C and D: UTM is conformal with a scale factor close to 1 (0.9996 on the central meridian): good for metric work in NRW.

## 4. Pitfalls you will meet
### 4.1 Axis order
EPSG:4326 officially defines **latitude first**. `pyproj.Transformer` respects that unless you ask for `always_xy=True`.

In [ ]:
t_official = Transformer.from_crs("EPSG:4326", "EPSG:25832")
t_xy = Transformer.from_crs("EPSG:4326", "EPSG:25832", always_xy=True)
print("lat, lon  ->", t_official.transform(a.y, a.x))
print("lon, lat  ->", t_xy.transform(a.x, a.y))
print("lon, lat without always_xy (wrong!) ->", t_official.transform(a.x, a.y))

### 4.2 Exercise: buffers
Create a **10 km buffer** around station 216 once in EPSG:4326 (with a radius in degrees you think is right) and once in EPSG:25832. Save both to `data/derived/buffers.gpkg` and compare them in QGIS. What shape does the "degree" buffer have, and why?

In [ ]:
# YOUR CODE HERE


## 5. Inspect a CRS like a developer

In [ ]:
crs = CRS("EPSG:25832")
print(crs.name)
print("axes:", [(ax.name, ax.unit_name) for ax in crs.axis_info])
print("area of use:", crs.area_of_use)
print(crs.to_wkt(pretty=True)[:600])

**Exercise (5 min):** compare `CRS("EPSG:25832")` with `CRS("EPSG:32632")`. Which parameters differ? Find them in the WKT. Then look at the false easting of the CRS you chose for file C.

## 6. QGIS check (view/QA)
1. *Layer → Add Layer → Delimited Text*: load A–D, each with the CRS you identified.
2. Set the project CRS to EPSG:25832 and add the NRW WMS (`https://www.wms.nrw.de/geobasis/wms_nw_dtk`).
3. All four layers must lie exactly on top of each other. Then load one file with a **wrong** CRS on purpose: where does it end up?
4. Load `data/derived/buffers.gpkg` and compare the two buffers.

## 7. Wrap-up
* **Exit ticket:** one sentence: what is the difference between a datum and a projection?
* **Homework and presentations:** see the course plan in the README of your course repository (it differs between courses).